# Fresh build — the VINE graph becomes the pipeline's graph

**No LLM. No API key.** A T4 is enough (an A100 is about three times faster).

What this notebook does:

| cell | what | time on a T4 |
|---|---|---|
| 1 | Drive, repo, packages | 3–5 min |
| 2 | **lists your Drive folder** and labels every item: source / rebuilt here / old / unknown | seconds |
| 3 | rebuilds the **VINE graph** from the manual and the repo code, and checks every fix | < 1 min |
| 4 | builds the table-row and figure-reading items | seconds |
| 5 | rebuilds the **text side of the vector store from scratch**: all 9,191 manual chunks + the items, freshly embedded, with every reference the graph found added to each chunk's payload | 15–25 min |
| 6 | saves the store to Drive | 1–2 min |
| 7 | loads the pipeline the normal way and shows it is running on the VINE graph | 1–2 min |
| 8 | moves the old files (labelled OLD in CELL 2) into `Beyond_RAG/_to_delete/` | seconds |

After this, `pipeline.kg` is the VINE graph for **everything** that uses a graph:
retrieval (search scoring, cross-references, notes, definitions, paragraphs), the
question router, per-check retrieval (Eq 5), and the cross-reference verifier — for the
RAG baseline and for VINE alike. The old `graph.gpickle` is not read by anything.

The image side of the store (page and figure-crop vectors) is kept as it is: those are
computed from the page images, which have not changed, so recomputing them would give
the same numbers after an hour of GPU time.

**Run every code cell in order: CELL 1 to CELL 8.** Then run `Retrieval_Test.ipynb`.

In [ ]:
# CELL 1 — Drive, repo, packages. MRAG_BASE_DIR must be set BEFORE any mrag import.
import sys, os
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_DIR = "/content/drive/MyDrive/Beyond_RAG"
REPO_DIR  = "/content/Beyond_RAG_repo"
REPO_URL  = "https://github.com/hannanazad/Beyond_RAG.git"

os.environ["MRAG_ENV"]      = "colab"
os.environ["MRAG_BASE_DIR"] = DRIVE_DIR

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull
sys.path.insert(0, REPO_DIR)

!pip install -q --index-url https://download.pytorch.org/whl/cu124 torch==2.6.0 torchvision==0.21.0
!pip install -q -r $REPO_DIR/requirements.txt
!pip install -q --no-deps --force-reinstall "transformers>=4.49,<4.55" "huggingface_hub>=0.34,<0.35" "tokenizers>=0.21,<0.22" "torchao>=0.13,<0.14"
!pip install -q pymupdf

for f in ("mrag/kg_vine.py", "mrag/vine/vector_items.py", "data/mutcd/FIGURE_READING_LOG.md"):
    assert os.path.exists(f"{REPO_DIR}/{f}"), f"{f} is not in the clone -- push the new files first"
print("data :", DRIVE_DIR)
print("code :", REPO_DIR)

### Your Drive folder, labelled
- **SOURCE** — read from the manual; everything is built from these. Keep.
- **REBUILT HERE** — this notebook writes a fresh copy.
- **RESULT** — a saved test run. Keep unless you decide otherwise.
- **OLD** — nothing reads it after this notebook. CELL 8 moves these to `_to_delete/`.
- **INGEST CACHE** — only the old ingest script reads it (embedding caches).
- **UNKNOWN** — not something the code reads or writes. Paste the list to Claude.

Nothing is moved or deleted in this cell.

In [ ]:
# CELL 2 — list everything in Beyond_RAG (and a sibling MRAG folder, if there is one).
import fnmatch, time
from pathlib import Path
from mrag.config import CFG

BASE = Path(CFG.base_dir)
RULES = [   # (label, glob relative to Beyond_RAG) -- first match wins
    ("SOURCE", "*.pdf"), ("SOURCE", "mmrag_cache_v3/chunks.jsonl"), ("SOURCE", "mmrag_cache_v3/figures.jsonl"),
    ("SOURCE", "mmrag_cache_v3/sign_codes.json"), ("SOURCE", "mmrag_cache_v3/mutcd_tables.jsonl"),
    ("SOURCE", "figures"), ("SOURCE", "page_images"),
    ("REBUILT HERE", "vine_data/graph_cache.pkl"), ("REBUILT HERE", "vine_data/vine_items.jsonl"),
    ("REBUILT HERE", "vine_data/text_store_manifest.json"), ("REBUILT HERE", "qdrant_db.tar"),
    ("RESULT", "retrieval_dev_results"),
    ("OLD", "mmrag_cache_v3/graph.gpickle"), ("OLD", "qdrant_db.before_*.tar"), ("OLD", "qdrant_db.writing.tar"),
    ("OLD", "vine_data/graph_cache.before_*.pkl"), ("OLD", "vine_data/graph_raw.pkl"),
    ("OLD", "vine_data/FIGURE_READING_LOG.md"), ("OLD", "vine_data/MANUAL_READING_LOG.md"),
    ("OLD", "vine_data/figure_rules.json"),
    ("INGEST CACHE", "mmrag_cache_v3/chunks_dense.npy"), ("INGEST CACHE", "mmrag_cache_v3/chunks_sparse.json"),
    ("INGEST CACHE", "mmrag_cache_v3/figures_dense.npy"), ("INGEST CACHE", "mmrag_cache_v3/text_embeddings_manifest.json"),
    ("INGEST CACHE", "mmrag_cache_v3/colqwen_pages"), ("INGEST CACHE", "mmrag_cache_v3/colqwen_figures"),
    ("INGEST CACHE", "mmrag_cache_v3/figure_coverage_report.json"), ("INGEST CACHE", "mmrag_cache_v3/chunks_v5.stamp"),
]

def label_of(rel: str) -> str:
    for lab, pat in RULES:
        if fnmatch.fnmatch(rel, pat):
            return lab
    return "UNKNOWN"

def size_of(p: Path) -> int:
    if p.is_file():
        return p.stat().st_size
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())

def entries():
    for p in sorted(BASE.iterdir()):
        if p.name == "_to_delete":
            continue
        if p.is_dir() and p.name in ("mmrag_cache_v3", "vine_data"):
            for q in sorted(p.iterdir()):
                yield q
        else:
            yield p

INVENTORY = []
for p in entries():
    rel = str(p.relative_to(BASE))
    INVENTORY.append((label_of(rel), rel, size_of(p), time.strftime("%Y-%m-%d", time.localtime(p.stat().st_mtime)),
                      "folder" if p.is_dir() else "file"))
order = ["SOURCE", "REBUILT HERE", "RESULT", "OLD", "INGEST CACHE", "UNKNOWN"]
for lab in order:
    rows = [r for r in INVENTORY if r[0] == lab]
    if rows:
        print(f"\n{lab}")
        for _, rel, sz, day, kind in rows:
            print(f"   {rel:60s} {sz / 1e6:10.1f} MB  {day}  {kind}")
sib = BASE.parent / "MRAG"
if sib.exists():
    print(f"\nUNKNOWN (sibling folder)\n   {sib}  {size_of(sib) / 1e6:10.1f} MB  -- an older project folder; nothing in the repo reads it")
print("\nPaste everything printed above to Claude before deleting anything labelled INGEST CACHE or UNKNOWN.")

### Rebuild the VINE graph
Inputs: `chunks.jsonl` and `mutcd_tables.jsonl` from Drive, the MUTCD PDF, and the three
reading files **from the repo** (`data/mutcd/`). The checks stop the notebook if a fix
did not take — for example if the corrected reading log was not pushed.

In [ ]:
# CELL 3 — rebuild the VINE graph from the manual and the repo code.
import json, pickle, shutil, subprocess
from collections import Counter
from mrag.vine.graph_links import build as build_sentence_graph, FHWA_2_09_2
from mrag.vine import graph_enrich

DATA_DIR = BASE / "vine_data"
GRAPH    = Path(CFG.vine_graph)
TABLES   = Path(CFG.tables_jsonl)
LOGS     = Path(REPO_DIR) / "data" / "mutcd"
DATA_DIR.mkdir(parents=True, exist_ok=True)
assert TABLES.exists(), f"missing {TABLES}"
assert CFG.pdf_path.exists(), f"missing the MUTCD PDF at {CFG.pdf_path}"

t0 = time.time()
chunks = [json.loads(l) for l in open(CFG.chunks_jsonl) if l.strip()]
N, E, cand = build_sentence_graph(chunks, tables_path=str(TABLES), interpretations=[FHWA_2_09_2])
print(f"step 1: {len(N):,} nodes, {len(E):,} edges")
graph_enrich.enrich(N, E, chunks, pdf=CFG.pdf_path,
                    figure_log=LOGS / "FIGURE_READING_LOG.md", rules=LOGS / "figure_rules.json",
                    reading_log=LOGS / "MANUAL_READING_LOG.md", tables=TABLES)
ok, tot = graph_enrich.resolution(N, E)
print(f"step 2: {len(N):,} nodes, {len(E):,} edges, {100 * ok / tot:.2f}% of edges resolve ({time.time() - t0:.0f}s)")
EXPECTED = (19671, 45488)    # what the same code gives from the same inputs
if (len(N), len(E)) != EXPECTED:
    print(f"NOTE: expected {EXPECTED[0]:,} nodes / {EXPECTED[1]:,} edges -- different inputs or code; look before going on")

out = {}
for e in E:
    out.setdefault(e.src, []).append(e)
gov = {e.why for e in out.get("trow:Table 2A-1#s1r6", []) if e.rel == "GOVERNED_BY"}
checks = {
    "Table 2C-4 chart A and chart B rows are separate":
        "trow:Table 2C-4#s1-Ar0" in N and N["trow:Table 2C-4#s1-Br0"].text == "5 mph",
    "table marks kept on their cells (Table 2A-1: Octagon*, Stop (R1-1)**)":
        "Shape=Octagon*" in N["trow:Table 2A-1#s1r0"].pieces and "Signs=Stop (R1-1)**" in N["trow:Table 2A-1#s1r0"].pieces,
    "a note applies only where its mark is printed (Diamond row: general note only)":
        gov == {"table note Note: whole table"},
    "column-heading marks apply to the whole column (Table 4C-1 a-d)":
        any("column" in e.why for e in out.get("trow:Table 4C-1#s1-Ar0", []) if e.rel == "GOVERNED_BY"),
    "no figure reading sits on a table of the same number":
        not any(e.rel == "HAS_READING" and e.src == "figure:Table 2C-4" for e in E),
    "'see Sections 2C.70 through 2C.73' links all four":
        {"section:2C.70", "section:2C.71", "section:2C.72", "section:2C.73"}
        <= {e.dst for e in out.get("sent:MUTCD11e_2C17_Option_03#0", [])},
    "the Notes for Figure 6P-10 link to the figure":
        any(e.dst == "figure:Figure 6P-10" for e in out.get("lead:MUTCD11e_6P01_TA10_Option_01", [])),
    "the question-wording speed words are gone":
        "posted at" not in open(f"{REPO_DIR}/mrag/vine/graph_links.py").read(),
    "the corrected figure readings are in (3B-28 reads from the hump outward)":
        "from the hump's leading edge outward" in open(LOGS / "FIGURE_READING_LOG.md").read(),
}
for name, passed in checks.items():
    print(f"  [{'ok' if passed else 'FAIL'}] {name}")
assert all(checks.values()), "a fix did not take -- is every file from the zip pushed, and the clone up to date?"

if GRAPH.exists():
    shutil.copy2(GRAPH, DATA_DIR / f"graph_cache.before_{time.strftime('%Y-%m-%d')}.pkl")
pickle.dump((N, E, chunks), open(GRAPH, "wb"))
print("saved", GRAPH)

In [ ]:
# CELL 4 — the items: one per figure reading, one per printed table row.
from mrag.vine.vector_items import build_items, write_items, summary, embed_text

items = build_items(GRAPH, TABLES)
ITEMS = Path(CFG.vine_items)
write_items(items, ITEMS)
print(summary(items))
print("written to", ITEMS)
for gid in ("trow:Table 2A-1#s1r0", "trow:Table 2M-1#s1r1"):
    it = next(i for i in items if i["graph_node"] == gid)
    print(f"\n{embed_text(it)[:420]}")

### A fresh text store
The chunk collection is **deleted and rebuilt**: every manual chunk and every item is
embedded again with BGE-M3. Each chunk's payload keeps everything it had and gains the
figure, table and section references the graph found (plural references, lead-in
references) — nothing is removed from a payload. The image collections are restored from
the last snapshot unchanged.

In [ ]:
# CELL 5 — rebuild the chunk collection from scratch.
from qdrant_client.http import models as qm
from mrag.embeddings import TextEmbedder
from mrag.vector_store import VectorStore, ChunkRow, chunk_id_to_int
from mrag.kg_vine import VineKG

LOCAL = Path(CFG.qdrant_dir)
SNAP  = BASE / "qdrant_db.tar"
if not (LOCAL / "collection" / CFG.coll_chunks).exists():
    assert SNAP.exists(), f"no store at {LOCAL} and no snapshot at {SNAP}"
    print(f"restoring {SNAP.name} for its image collections ...")
    if LOCAL.exists():
        shutil.rmtree(LOCAL)
    subprocess.run(["tar", "-xf", str(SNAP), "-C", str(LOCAL.parent)], check=True)

kg = VineKG(GRAPH, figures_path=CFG.figures_jsonl, sign_codes_path=CFG.sign_codes_json,
            items_path=ITEMS, tables_path=TABLES)
added = Counter()
payloads = []
for c in chunks:
    p = dict(c)
    refs = kg.references_of_chunk(c["chunk_id"])
    for k in ("figure_refs", "table_refs", "section_refs"):
        have = list(p.get(k) or [])
        new = [x for x in refs[k] if x not in have]
        added[k] += len(new)
        p[k] = have + new                         # keep every original reference, add the graph's
    payloads.append(p)
print("references added to chunk payloads from the graph:", dict(added))

def chunk_text(c):          # exactly as scripts/ingest_v3.py embeds a chunk
    return f"[{c['content_type']}] Section {c['section_id']} — {c['section_title']}. {c['text']}"

texts = [chunk_text(c) for c in payloads] + [embed_text(i) for i in items]
all_payloads = payloads + items
assert len({p["chunk_id"] for p in all_payloads}) == len(all_payloads), "duplicate ids"

te = TextEmbedder(CFG.bge_m3_model).load()
print("encoder mode:", te._mode)
t0 = time.time()
dense, sparse = te.encode_both(texts, batch_size=32)
print(f"embedded {len(texts):,} texts in {time.time() - t0:.0f}s | with no sparse weights: {sum(1 for s in sparse if not s)}")

store = VectorStore(LOCAL)
cl = store.client
if cl.collection_exists(CFG.coll_chunks):
    cl.delete_collection(CFG.coll_chunks)
cl.create_collection(collection_name=CFG.coll_chunks,
                     vectors_config={"dense": qm.VectorParams(size=dense.shape[1], distance=qm.Distance.COSINE)},
                     sparse_vectors_config={"sparse": qm.SparseVectorParams()})
store.upsert_chunks(CFG.coll_chunks, [ChunkRow(id=chunk_id_to_int(p["chunk_id"]), dense=d, sparse=s, payload=p)
                                      for p, d, s in zip(all_payloads, dense, sparse)])
n_store = cl.count(CFG.coll_chunks, exact=True).count
print(f"chunk collection: {n_store:,} = {len(payloads):,} manual chunks + {len(items):,} items")
assert n_store == len(all_payloads)
print("other collections kept:", [c.name for c in cl.get_collections().collections if c.name != CFG.coll_chunks])
json.dump({"built": time.strftime("%Y-%m-%d %H:%M"), "encoder_mode": te._mode, "chunks": len(payloads),
           "items": len(items), "graph": str(GRAPH), "graph_nodes": len(N), "graph_edges": len(E),
           "references_added": dict(added)},
          open(DATA_DIR / "text_store_manifest.json", "w"), indent=1)
store.close()
del te

In [ ]:
# CELL 6 — save the store to Drive; the previous snapshot is kept, by name, for CELL 8.
TMP = BASE / "qdrant_db.writing.tar"
OLD = BASE / f"qdrant_db.before_{time.strftime('%Y-%m-%d_%H%M')}.tar"   # never overwrites an earlier one
t0 = time.time()
subprocess.run(["tar", "-cf", str(TMP), "-C", str(LOCAL.parent), LOCAL.name], check=True)
if SNAP.exists():
    SNAP.rename(OLD)
    print("previous snapshot kept as", OLD.name, "(CELL 8 moves it to _to_delete)")
TMP.rename(SNAP)
print(f"saved {SNAP} ({SNAP.stat().st_size / 1e6:.0f} MB) in {time.time() - t0:.0f}s")

In [ ]:
# CELL 7 — load the pipeline the normal way (no LLM, no image model) and look at it.
from mrag.ask import init_pipeline
pipeline = init_pipeline(load_image_embedder=False, load_vlm=False)
print("graph class :", type(pipeline.kg).__name__, "| from", pipeline.kg.g.graph.get("source"))
print("graph size  :", pipeline.kg.g.graph.get("build_stats"))
print("4K.04 cites :", pipeline.kg.sections_cited_by("4K.04"))
print("2C.17 cites :", pipeline.kg.sections_cited_by("2C.17"))
res = pipeline.retriever.retrieve_for_compile("minimum mounting height of a sign above a sidewalk")
print("Kq size     :", len(res.chunks), "| sources:", dict(Counter(c.get("source") for c in res.chunks)))
print("item types  :", dict(Counter(c.get("content_type") for c in res.chunks)))

### Move the old files out of the way
Only items CELL 2 labelled **OLD** are moved, into `Beyond_RAG/_to_delete/`. Nothing is
deleted. When you are satisfied, delete that folder from Drive yourself. **INGEST CACHE**
and **UNKNOWN** items are left where they are until you decide.

In [ ]:
# CELL 8 — move the OLD items into _to_delete/.
BIN = BASE / "_to_delete"
BIN.mkdir(exist_ok=True)
moved = []
for p in list(entries()):
    rel = str(p.relative_to(BASE))
    if label_of(rel) == "OLD" and p.exists():
        dst = BIN / rel.replace("/", "__")
        shutil.move(str(p), str(dst))
        moved.append(rel)
print(f"moved {len(moved)} item(s) to {BIN}:")
for m in moved:
    print("  ", m)
print("\nNext: run Retrieval_Test.ipynb, CELL 1 to CELL 8.")